# 使用 `polmeas` 执行 CHSH Bell scan

这个 Notebook 是一个真实硬件测量模板：它从当前工作目录的 `polmeas_config.yaml` 读取电机和 TDC 配置，扫描 Bob 分析基的偏移角 `β`，并计算 CHSH Bell 参数 `S(β)`。

运行前请确认：

- 已从仓库根目录安装 `qtools`（例如 `uv sync` 或 `pip install -e .`）；
- 当前工作目录中存在已经配置好的 `polmeas_config.yaml`；
- 两个偏振分析臂的电机和 TDC 已连接并完成必要校准。

本示例不会创建模拟设备；没有硬件或配置错误时，测量单元会直接抛出 `qtools.pmeas` 的原始异常。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qtools.pmeas import TaskController

# `polmeas_config.yaml` 中保存电机、TDC、通道和校准参数。
CONFIG_PATH = Path("polmeas_config.yaml")

# Bell scan 参数，可按实验的计数率和积分时间调整。
START_OFFSET = 0.0
END_OFFSET = 90.0
STEP_OFFSET = 5.0
DURATION = 1.0
REPETITIONS = 3
CHANNELS = (0, 1)
SUBTRACT_ACCIDENTALS = True
SAVE_RAW = True
SAVE_PLOT = False


## 执行 Bell scan

`TaskController` 负责打开和关闭配置中的电机与 TDC。`scan_bell` 会在每个 `β` 偏移位置测量 CHSH 所需的四组基组合，并返回扫描曲线和误差。

In [ ]:
with TaskController(config_path=str(CONFIG_PATH)) as ctrl:
    bell_result = ctrl.scan_bell(
        start_offset=START_OFFSET,
        end_offset=END_OFFSET,
        step_offset=STEP_OFFSET,
        duration=DURATION,
        repetitions=REPETITIONS,
        subtract_accidentals=SUBTRACT_ACCIDENTALS,
        channel=CHANNELS,
        save_raw=SAVE_RAW,
        save_plot=SAVE_PLOT,
    )

print("Bell scan finished.")


In [ ]:
# 结果字典中的数组可继续用于拟合或保存到其他数据格式。
offsets = np.asarray(bell_result["offsets"])
s_values = np.asarray(bell_result["s_values"])
s_errors = np.asarray(bell_result["s_errors"])

print(f"最大 |S|: {abs(bell_result['max_s']):.4f} ± {bell_result['max_s_err']:.4f}")
print(f"对应 Bob 偏移 β: {bell_result['max_offset']:.2f}°")


## 绘制 `S(β)`

经典局域隐变量理论要求 `|S| ≤ 2`。图中的误差棒来自各个相关函数的测量误差传播。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# 经典 Bell 界限区域和边界。
ax.axhspan(-2, 2, color="0.85", alpha=0.6, label=r"classical region $|S|\leq 2$")
ax.axhline(2, color="0.35", linestyle="--", linewidth=1)
ax.axhline(-2, color="0.35", linestyle="--", linewidth=1)

ax.errorbar(
    offsets,
    s_values,
    yerr=s_errors,
    fmt="o-",
    capsize=3,
    color="tab:blue",
    label=r"measured $S(\beta)$",
)

max_index = int(np.argmax(np.abs(s_values)))
ax.scatter(
    offsets[max_index],
    s_values[max_index],
    s=90,
    color="tab:red",
    zorder=3,
    label=f"max |S| = {abs(s_values[max_index]):.3f}",
)

ax.set_xlabel(r"Bob basis offset $\beta$ (degrees)")
ax.set_ylabel("CHSH Bell parameter S")
ax.set_title("CHSH Bell scan")
ax.grid(True, linestyle=":", alpha=0.7)
ax.legend()
fig.tight_layout()
plt.show()


## 返回结果

`bell_result` 包含以下字段：

- `offsets`：扫描的 Bob 偏移角；
- `s_values`：每个偏移角对应的 CHSH 参数；
- `s_errors`：`S` 的不确定度；
- `max_s`、`max_s_err`、`max_offset`：最大 `|S|` 及其位置。

当 `SAVE_RAW = True` 时，`polmeas` 还会按照当前输出目录配置保存一个 `bell_raw_*.npz` 文件；是否保存图片由 `SAVE_PLOT` 控制。